# **WEEK 3 — Forecasting Theory (Not Just Point Forecasts)**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Ch 3.4–3.7: Forecasting)

---

**Purpose of this Notebook**

This notebook develops forecasting as a **theory of prediction under uncertainty**,
not merely the generation of point forecasts.

**The goal is to understand:**

- $h$-step forecasting rules for ARMA-type models
- how forecast error variance accumulates with horizon
- how prediction intervals are constructed and what breaks them
- model misspecification effects on forecast quality
- backtesting discipline with rolling-origin evaluation

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Compute analytic 1-step and $h$-step forecasts for AR(1) and ARMA(1,1)
2. Derive forecast error variance and construct prediction intervals
3. Explain $\psi$-weights and their role in forecast uncertainty
4. Identify what breaks prediction intervals (misspecification, heavy tails, heteroskedasticity)
5. Implement rolling-origin backtesting and compare against naive baselines
6. Defend a forecasting approach in an exam setting

---

**Notebook Structure**

| Part | Topic | Type |
|------|-------|------|
| 1 | Forecasting Philosophy | Theory |
| 2 | AR(1) Analytic Forecasts and Error Variance | Theory + Simulation |
| 3 | ARMA(1,1) Forecast Error Variance ($\psi$-weights) | Theory + Simulation |
| 4 | Prediction Intervals and Fan Charts | Theory + Simulation |
| 5 | What Breaks Prediction Intervals | Theory + Simulation |
| 6 | Model Misspecification Effects | Theory + Simulation |
| 7 | Backtesting with Rolling Origin | Theory + Simulation |
| 8 | Synthesis (Exam-Ready) | Summary |
| 9 | Self-Test Questions | Exam Preparation |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(123)

# --- Shared AR(1) simulator ---
def simulate_ar1(phi, sigma, n=300, burnin=200):
    eps = np.random.normal(0, sigma, size=n + burnin)
    x = np.zeros(n + burnin)
    for t in range(1, n + burnin):
        x[t] = phi * x[t-1] + eps[t]
    return x[burnin:]

# --- Shared ARMA(1,1) simulator ---
def simulate_arma11(phi, theta, sigma, n=400, burnin=200, shock="gaussian", df=3):
    N = n + burnin + 1
    if shock == "gaussian":
        eps = np.random.normal(0, sigma, N)
    elif shock == "t":
        z = student_t.rvs(df, size=N)
        z = z / np.sqrt(df / (df - 2))  # standardize to unit variance
        eps = sigma * z
    else:
        raise ValueError("shock must be 'gaussian' or 't'")
    x = np.zeros(N)
    for t in range(1, N):
        x[t] = phi * x[t-1] + eps[t] + theta * eps[t-1]
    return x[burnin+1:]

print("Setup complete.")


## **PART 1 — Forecasting as Best Prediction + Quantified Uncertainty**

> **Core idea (Shumway & Stoffer)**
>
> Forecasting is not only about producing a point estimate.
> It is also about **quantifying uncertainty** through forecast error variance.

---

### **1.1 Two Objects Define a Forecast**

- **Point forecast:** $\hat{x}_{t+h|t}$ — the best linear prediction of $X_{t+h}$ given information up to time $t$
- **Forecast error:** $e_{t+h} = X_{t+h} - \hat{x}_{t+h|t}$

A forecast is meaningful only when we can characterize:

- $\mathbb{E}[e_{t+h}]$ — bias (should be zero for optimal forecasts)
- $\mathrm{Var}(e_{t+h})$ — uncertainty growth with horizon

---

### **1.2 Prediction Intervals**

Prediction intervals are built from the **forecast distribution**:

$$
X_{t+h} \mid \mathcal{F}_t \;\sim\; \text{Normal}\!\left(\hat{x}_{t+h|t},\; \mathrm{Var}(e_{t+h})\right)
$$

A $(1-\alpha)$ prediction interval:

$$
\hat{x}_{t+h|t} \;\pm\; z_{1-\alpha/2} \cdot \sqrt{\mathrm{Var}(e_{t+h})}
$$

These intervals **require assumptions** and they **break when assumptions fail**.

> This notebook begins with AR(1) because it has closed-form $h$-step forecasts and
> closed-form error variance. We then generalize to ARMA via $\psi$-weights.


## **PART 2 — AR(1) Analytic Forecasts and Forecast Error Variance**

> **Model:**
>
> $$X_t = \phi X_{t-1} + \varepsilon_t, \qquad \varepsilon_t \sim \mathrm{WN}(0, \sigma^2), \quad |\phi| < 1$$

---

### **2.1 One-step-ahead forecast**

$$
\hat{x}_{t+1|t} = \phi X_t
$$

Forecast error: $e_{t+1} = \varepsilon_{t+1}$, so $\mathrm{Var}(e_{t+1}) = \sigma^2$.

---

### **2.2 $h$-step-ahead forecast**

By repeated substitution:

$$
\hat{x}_{t+h|t} = \phi^h X_t
$$

The point forecast **decays exponentially** toward the unconditional mean (zero).

---

### **2.3 Forecast error variance ($h$-step)**

The $h$-step forecast error accumulates future shocks:

$$
e_{t+h} = \varepsilon_{t+h} + \phi\,\varepsilon_{t+h-1} + \cdots + \phi^{h-1}\varepsilon_{t+1}
$$

Therefore:

$$
\boxed{\mathrm{Var}(e_{t+h}) = \sigma^2 \sum_{j=0}^{h-1} \phi^{2j} = \sigma^2 \cdot \frac{1 - \phi^{2h}}{1 - \phi^2}}
$$

As $h \to \infty$: $\mathrm{Var}(e_{t+h}) \to \sigma^2 / (1 - \phi^2) = \gamma(0)$ (unconditional variance).

---

### **2.4 Prediction interval**

$$
\hat{x}_{t+h|t} \;\pm\; z_{1-\alpha/2} \cdot \sigma \sqrt{\frac{1 - \phi^{2h}}{1 - \phi^2}}
$$

> **Key insight:** Uncertainty grows with horizon and saturates at the unconditional variance.
> This is the fundamental reason forecasts become uninformative at long horizons.


In [ ]:
# ============================================================
# PART 2: AR(1) Analytic Forecasts + Error Variance
# ============================================================

def ar1_point_forecast(phi, x_t, h):
    return (phi ** h) * x_t

def ar1_forecast_var(phi, sigma, h):
    return (sigma**2) * (1 - phi**(2*h)) / (1 - phi**2)

def ar1_prediction_interval(phi, sigma, x_t, h, alpha=0.05):
    mu = ar1_point_forecast(phi, x_t, h)
    var = ar1_forecast_var(phi, sigma, h)
    z = norm.ppf(1 - alpha/2)
    lo = mu - z * np.sqrt(var)
    hi = mu + z * np.sqrt(var)
    return mu, lo, hi, var

# --- Parameters ---
phi = 0.7
sigma = 1.0
n = 350
x = simulate_ar1(phi, sigma, n=n)
x_t = x[-1]  # last observed value

# --- Tabulate forecasts ---
hs = [1, 3, 6, 12, 24]
print(f"AR(1) with phi={phi}, sigma={sigma}")
print(f"Last observed X_t = {x_t:.3f}")
print(f"\n{'h':>4s}  {'Forecast':>10s}  {'Var(e)':>10s}  {'95% PI':>24s}")
print("-" * 55)
for h in hs:
    mu, lo, hi, var = ar1_prediction_interval(phi, sigma, x_t, h)
    print(f"{h:4d}  {mu:10.3f}  {var:10.3f}  ({lo:10.3f}, {hi:10.3f})")

# --- Forecast error variance growth ---
Hs = np.arange(1, 51)
vars_h = np.array([ar1_forecast_var(phi, sigma, h) for h in Hs])
unconditional_var = sigma**2 / (1 - phi**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Variance growth
axes[0].plot(Hs, vars_h, linewidth=2, color="#2c3e50")
axes[0].axhline(unconditional_var, color="red", linestyle="--", alpha=0.7,
                label=f"$\\gamma(0) = \\sigma^2/(1-\\phi^2) = {unconditional_var:.2f}$")
axes[0].set_title(f"AR(1) Forecast Error Variance ($\\phi={phi}$)", fontsize=12)
axes[0].set_xlabel("Horizon $h$")
axes[0].set_ylabel("$\\mathrm{Var}(e_{t+h})$")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Compare different phi values
for phi_val, color in [(0.3, "#3498db"), (0.7, "#2c3e50"), (0.95, "#e74c3c")]:
    v = [ar1_forecast_var(phi_val, sigma, h) for h in Hs]
    uv = sigma**2 / (1 - phi_val**2)
    axes[1].plot(Hs, v, linewidth=2, color=color, label=f"$\\phi={phi_val}$")
    axes[1].axhline(uv, color=color, linestyle=":", alpha=0.4)

axes[1].set_title("Variance Growth by $\\phi$", fontsize=12)
axes[1].set_xlabel("Horizon $h$")
axes[1].set_ylabel("$\\mathrm{Var}(e_{t+h})$")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observation:")
print("  Higher phi -> slower saturation -> forecasts stay informative longer")
print("  All converge to unconditional variance (horizontal dashed lines)")


## **PART 3 — ARMA(1,1) Forecast Error Variance ($\psi$-weights)**

> **Why this part matters**
>
> AR(1) has clean closed forms. But the exam expects you to understand the **general
> ARMA logic**: forecasting depends on the **innovation representation** and the
> **$\psi$-weights** (Shumway & Stoffer, Ch. 3.4).

---

### **3.1 Model**

$$
X_t = \phi X_{t-1} + \varepsilon_t + \theta \varepsilon_{t-1}
$$

---

### **3.2 Wold / MA($\infty$) representation and $\psi$-weights**

Under stationarity ($|\phi| < 1$) and invertibility, ARMA(1,1) admits:

$$
X_t = \sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j}
$$

For ARMA(1,1), the $\psi$-weights are:

$$
\psi_0 = 1, \qquad \psi_j = \phi^{j-1}(\phi + \theta) \;\text{ for } j \geq 1
$$

> These are the **bridge** between model structure and forecast uncertainty.

---

### **3.3 Forecast error variance ($h$-step)**

$$
e_{t+h} = \varepsilon_{t+h} + \psi_1 \varepsilon_{t+h-1} + \cdots + \psi_{h-1}\varepsilon_{t+1}
$$

$$
\boxed{\mathrm{Var}(e_{t+h}) = \sigma^2 \left[1 + \sum_{j=1}^{h-1} \psi_j^2\right] = \sigma^2 \left[1 + (\phi+\theta)^2 \cdot \frac{1 - \phi^{2(h-1)}}{1 - \phi^2}\right]}
$$

---

### **3.4 Prediction interval (conditional Normal assumption)**

$$
\hat{x}_{t+h|t} \;\pm\; z_{1-\alpha/2} \cdot \sqrt{\mathrm{Var}(e_{t+h})}
$$

> **Exam-safe statement:** "ARMA prediction intervals are justified through $\psi$-weights.
> When shocks are not Gaussian or variance is not constant, these intervals can be
> systematically wrong even if the point forecast is reasonable."


In [ ]:
# ============================================================
# PART 3: ARMA(1,1) psi-weights and Forecast Error Variance
# ============================================================

def arma11_psi_weights(phi, theta, H):
    """Return psi_0..psi_{H-1} for ARMA(1,1)."""
    psi = np.zeros(H)
    psi[0] = 1.0
    if H >= 2:
        psi[1] = phi + theta
    for j in range(2, H):
        psi[j] = (phi ** (j-1)) * (phi + theta)
    return psi

def arma11_forecast_var(phi, theta, sigma, h):
    """Var(e_{t+h}) for ARMA(1,1)."""
    if h == 1:
        return sigma**2
    return (sigma**2) * (1 + (phi + theta)**2 * (1 - phi**(2*(h-1))) / (1 - phi**2))

# --- Parameters ---
phi, theta, sigma = 0.6, 0.7, 1.0

# --- Compare AR(1) vs ARMA(1,1) variance growth ---
Hs = np.arange(1, 41)
var_ar1 = np.array([ar1_forecast_var(phi, sigma, h) for h in Hs])
var_arma = np.array([arma11_forecast_var(phi, theta, sigma, h) for h in Hs])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Variance comparison
axes[0].plot(Hs, var_ar1, linewidth=2, label=f"AR(1) $\\phi={phi}$", color="#3498db")
axes[0].plot(Hs, var_arma, linewidth=2, label=f"ARMA(1,1) $\\phi={phi}, \\theta={theta}$", color="#e74c3c")
axes[0].set_title("Forecast Error Variance: AR(1) vs ARMA(1,1)", fontsize=12)
axes[0].set_xlabel("Horizon $h$")
axes[0].set_ylabel("$\\mathrm{Var}(e_{t+h})$")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Psi-weights
H = 20
psi = arma11_psi_weights(phi, theta, H)
axes[1].stem(range(H), psi, basefmt=" ", markerfmt="o", linefmt="-")
axes[1].set_title(f"ARMA(1,1) $\\psi$-weights ($\\phi={phi}, \\theta={theta}$)", fontsize=12)
axes[1].set_xlabel("$j$")
axes[1].set_ylabel("$\\psi_j$")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ARMA(1,1) psi-weights (first 10):")
for j in range(10):
    print(f"  psi_{j} = {psi[j]:.4f}")
print(f"\nNote: psi_1 = phi + theta = {phi} + {theta} = {phi+theta}")
print(f"The MA component amplifies initial uncertainty, then AR decay takes over.")


## **PART 4 — Prediction Intervals and Fan Charts**

> **Why this part matters**
>
> A fan chart visualizes how prediction uncertainty **widens with horizon**.
> It is the visual representation of $\mathrm{Var}(e_{t+h})$ growing with $h$.
>
> Understanding fan charts is essential for communicating forecast uncertainty honestly.

---

### **4.1 Fan Chart Construction**

For each horizon $h = 1, 2, \ldots, H$:

1. Compute point forecast: $\hat{x}_{t+h|t}$
2. Compute forecast error variance: $\mathrm{Var}(e_{t+h})$
3. Compute prediction interval: $\hat{x}_{t+h|t} \pm z_{\alpha/2} \sqrt{\mathrm{Var}(e_{t+h})}$

Plot the point forecasts as a line and shade the prediction intervals.
Wider bands = more uncertainty.

> **Exam-safe statement:** "Fan charts communicate that uncertainty grows with horizon.
> When the bands saturate at $\gamma(0)$, the model provides no information beyond the unconditional distribution."


In [ ]:
# ============================================================
# PART 4: Fan Chart — AR(1) h-step Prediction Intervals
# ============================================================

np.random.seed(123)

phi, sigma = 0.7, 1.0
n = 200
x = simulate_ar1(phi, sigma, n=n)
x_t = x[-1]

# --- Compute h-step forecasts and PIs ---
H_max = 40
horizons = np.arange(1, H_max + 1)
forecasts = np.array([ar1_point_forecast(phi, x_t, h) for h in horizons])
variances = np.array([ar1_forecast_var(phi, sigma, h) for h in horizons])
stds = np.sqrt(variances)

# --- Fan chart ---
fig, ax = plt.subplots(1, 1, figsize=(14, 5))

# Plot observed series (last 60 points)
tail = 60
t_obs = np.arange(n - tail, n)
ax.plot(t_obs, x[-tail:], linewidth=1.5, color="#2c3e50", label="Observed")

# Plot forecasts
t_fc = np.arange(n, n + H_max)
ax.plot(t_fc, forecasts, linewidth=2, color="#e74c3c", label="Point forecast")

# Prediction intervals at 50%, 80%, 95%
for level, alpha_val, shade in [(0.50, 0.50, 0.30), (0.80, 0.20, 0.20), (0.95, 0.05, 0.12)]:
    z = norm.ppf(1 - alpha_val / 2)
    lo = forecasts - z * stds
    hi = forecasts + z * stds
    ax.fill_between(t_fc, lo, hi, alpha=shade, color="#e74c3c", label=f"{int(level*100)}% PI")

ax.axvline(n - 0.5, color="gray", linestyle="--", alpha=0.5, label="Forecast origin")
ax.set_title(f"AR(1) Fan Chart ($\\phi={phi}$, $\\sigma={sigma}$)", fontsize=13)
ax.set_xlabel("Time")
ax.set_ylabel("$X_t$")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations:")
print("  - PI bands widen with horizon (uncertainty grows)")
print("  - Bands saturate as Var(e) -> gamma(0)")
print(f"  - Unconditional variance: gamma(0) = {sigma**2/(1-phi**2):.3f}")
print(f"  - At h={H_max}: Var(e) = {variances[-1]:.3f} (nearly saturated)")
print("  - Point forecast decays toward 0 (unconditional mean)")


## **PART 5 — What Breaks Prediction Intervals**

> **Prediction intervals are conditional claims.** They rely on assumptions
> that are frequently violated in real time series.

---

### **5.1 What a Gaussian PI Assumes**

A standard ARMA PI implicitly assumes:

- Model is **correctly specified** (right $p, d, q$ orders)
- Parameters ($\phi, \theta, \sigma^2$) are **known** or estimated with negligible error
- Shocks are **independent** (or at least uncorrelated)
- **Constant variance** (homoskedasticity)
- **No structural breaks** or regime shifts
- Correct differencing order (if ARIMA)

---

### **5.2 Common Failure Modes**

| Failure | Effect on PIs |
|---|---|
| **Misspecification** (wrong ARMA order, missing seasonality) | Biased variance estimates, systematic over/undercoverage |
| **Nonstationarity / structural change** | Drift invalidates fixed-parameter PIs |
| **Heteroskedasticity** (ARCH/GARCH) | PIs too narrow in volatile periods, too wide in calm ones |
| **Non-Gaussian shocks** (heavy tails) | Systematic undercoverage (extreme values fall outside bands) |
| **Parameter uncertainty** | Estimated $\hat{\phi}, \hat{\sigma}^2$ add unaccounted uncertainty |

---

### **5.3 Empirical Coverage**

The **empirical coverage** of a nominal $(1-\alpha)$ PI is:

$$
\hat{C} = \frac{1}{N}\sum_{i=1}^{N} \mathbf{1}\!\left\{X_{t_i+h} \in \left[\hat{x}_{t_i+h|t_i}^{\text{lo}},\; \hat{x}_{t_i+h|t_i}^{\text{hi}}\right]\right\}
$$

If $\hat{C} \ll 1 - \alpha$: intervals are too narrow (undercoverage).
If $\hat{C} \gg 1 - \alpha$: intervals are too wide (overcoverage, conservative).

> **Exam-safe statement:** "Passing a whiteness test does not guarantee correct interval coverage.
> Interval validity is a **stronger claim** than residual uncorrelatedness."


In [ ]:
# ============================================================
# PART 5: PI Failure Demo — Gaussian vs Heavy-Tailed Shocks
# ============================================================

np.random.seed(123)

phi_true, theta_true, sigma_true = 0.6, 0.7, 1.0
n = 600
initial_train = 300
alpha = 0.05

# --- Simulate two DGPs ---
x_gauss = simulate_arma11(phi_true, theta_true, sigma_true, n=n, shock="gaussian")
x_ttail = simulate_arma11(phi_true, theta_true, sigma_true, n=n, shock="t", df=3)

# --- Rolling one-step PI coverage ---
def rolling_one_step_pi_coverage(y, order, initial_train=250, alpha=0.05):
    y = pd.Series(y).dropna().reset_index(drop=True)
    n = len(y)
    train = y.iloc[:initial_train]
    res = ARIMA(train, order=order).fit()
    records = []
    for i in range(initial_train, n):
        fc = res.get_forecast(steps=1)
        mean_val = float(fc.predicted_mean.iloc[0])
        ci = fc.conf_int(alpha=alpha)
        lo = float(ci.iloc[0, 0])
        hi = float(ci.iloc[0, 1])
        actual = float(y.iloc[i])
        covered = (lo <= actual <= hi)
        records.append({"t": i, "actual": actual, "mean": mean_val,
                        "lo": lo, "hi": hi, "covered": covered})
        res = res.append([actual], refit=False)
    out = pd.DataFrame(records)
    return out["covered"].mean(), out

# --- Compare correct vs misspecified model ---
configs = [
    ("ARMA(1,1) — correct", (1, 0, 1)),
    ("AR(1) — misspecified", (1, 0, 0)),
]

results = []
for label, order in configs:
    cov_g, _ = rolling_one_step_pi_coverage(x_gauss, order=order, initial_train=initial_train, alpha=alpha)
    cov_t, _ = rolling_one_step_pi_coverage(x_ttail, order=order, initial_train=initial_train, alpha=alpha)
    results.append({"Model": label, "Coverage (Gaussian)": f"{cov_g:.3f}", "Coverage (t, df=3)": f"{cov_t:.3f}"})

print("=== Empirical Coverage of Nominal 95% One-Step PIs ===")
print(pd.DataFrame(results).to_string(index=False))

# --- Plot PI for visual inspection ---
_, out_g = rolling_one_step_pi_coverage(x_gauss, order=(1,0,1), initial_train=300, alpha=0.05)
_, out_t = rolling_one_step_pi_coverage(x_ttail, order=(1,0,1), initial_train=300, alpha=0.05)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
seg = 100

for ax, out, title in [
    (axes[0], out_g, "Gaussian DGP — ARMA(1,1) fit"),
    (axes[1], out_t, "Heavy-tailed DGP (t, df=3) — ARMA(1,1) fit")
]:
    tail = out.tail(seg)
    cov = tail["covered"].mean()
    ax.plot(tail["t"], tail["actual"], linewidth=1.5, color="#2c3e50", label="Actual")
    ax.plot(tail["t"], tail["mean"], linewidth=1.0, color="#e74c3c", label="Forecast")
    ax.fill_between(tail["t"], tail["lo"], tail["hi"], alpha=0.25, color="#3498db", label="95% PI")
    # Mark misses
    misses = tail[~tail["covered"]]
    ax.scatter(misses["t"], misses["actual"], color="red", s=20, zorder=5, label="PI miss")
    ax.set_title(f"{title} — coverage = {cov:.3f}", fontsize=11)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Gaussian DGP + correct model: coverage ~ 0.95 (nominal)")
print("  - Heavy-tailed DGP: coverage < 0.95 (undercoverage, more red dots)")
print("  - Misspecification also degrades coverage")
print("  -> PI validity requires BOTH correct model AND correct distributional assumptions")


## **PART 6 — Model Misspecification: What It Does to Forecasting**

> **Core idea**
>
> Forecasting performance and prediction interval validity depend on **correct structure**.
> A common misspecification is fitting AR(1) to a process with additional MA dynamics
> (e.g., ARMA(1,1)).

---

### **6.1 Effects of Misspecification**

Even when a fitted AR(1) seems "reasonable":

- Point forecasts may look acceptable (bias is small at short horizons)
- But forecast error variance is **wrong** (based on incorrect $\psi$-weights)
- Prediction intervals have **incorrect width**
- Coverage degrades, especially at longer horizons

---

### **6.2 Why This Is Exam-Critical**

The examiner may ask: *"What happens if you fit AR(1) to an ARMA(1,1) process?"*

Correct answer:
- The AR(1) captures the dominant autoregressive structure
- But it **misses the MA component**, leading to misestimated one-step error variance
- Residuals may still appear approximately white (especially in finite samples)
- However, PI coverage will be systematically off


In [ ]:
# ============================================================
# PART 6: Misspecification — AR(1) fit to ARMA(1,1) DGP
# ============================================================

np.random.seed(123)

# True DGP: ARMA(1,1)
phi_true, theta_true, sigma_true = 0.6, 0.7, 1.0
x_arma = simulate_arma11(phi_true, theta_true, sigma_true, n=500, burnin=200)

# --- ACF/PACF of the ARMA(1,1) process ---
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

axes[0].plot(x_arma, linewidth=0.8, color="#2c3e50")
axes[0].set_title("ARMA(1,1) Series (True DGP)", fontsize=11)
axes[0].grid(True, alpha=0.3)

a = acf(x_arma, nlags=30, fft=True)
p_vals = np.array([0] + list(acf(x_arma, nlags=30, fft=True)))  # placeholder for PACF
from statsmodels.tsa.stattools import pacf as compute_pacf
p_vals = compute_pacf(x_arma, nlags=30, method="ywm")

ci = 1.96 / np.sqrt(len(x_arma))
axes[1].stem(range(len(a)), a, basefmt=" ")
axes[1].axhline(ci, color="red", linestyle="--", alpha=0.5)
axes[1].axhline(-ci, color="red", linestyle="--", alpha=0.5)
axes[1].set_title("ACF of ARMA(1,1)", fontsize=11)
axes[1].grid(True, alpha=0.3)

axes[2].stem(range(len(p_vals)), p_vals, basefmt=" ")
axes[2].axhline(ci, color="red", linestyle="--", alpha=0.5)
axes[2].axhline(-ci, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("PACF of ARMA(1,1)", fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Fit both correct and misspecified models ---
res_correct = ARIMA(x_arma, order=(1, 0, 1)).fit()
res_misspec = ARIMA(x_arma, order=(1, 0, 0)).fit()

print("=== Model Comparison ===")
print(f"{'Model':<20s}  {'AIC':>10s}  {'BIC':>10s}")
print(f"{'ARMA(1,1) (correct)':<20s}  {res_correct.aic:10.2f}  {res_correct.bic:10.2f}")
print(f"{'AR(1) (misspecified)':<20s}  {res_misspec.aic:10.2f}  {res_misspec.bic:10.2f}")

# --- Residual ACF comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
ci = 1.96 / np.sqrt(len(x_arma))

for ax, res, label in [
    (axes[0], res_correct, "ARMA(1,1) residuals"),
    (axes[1], res_misspec, "AR(1) residuals (misspecified)")
]:
    r = res.resid.dropna()
    a_r = acf(r, nlags=30, fft=True)
    ax.stem(range(len(a_r)), a_r, basefmt=" ")
    ax.axhline(ci, color="red", linestyle="--", alpha=0.5)
    ax.axhline(-ci, color="red", linestyle="--", alpha=0.5)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Lag")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nExam note:")
print("  ARMA(1,1) ACF/PACF can produce ambiguous patterns")
print("  AR(1) residuals may show a spike at lag 1 (missed MA component)")
print("  AIC correctly prefers ARMA(1,1), but the key lesson is about PI coverage (Part 5)")


## **PART 7 — Backtesting with Rolling Origin**

> **Why rolling origin is required**
>
> Time series forecasting must **respect time order**. You cannot use future data
> to evaluate past forecasts.
>
> Rolling-origin backtesting evaluates a forecasting rule under repeated,
> out-of-sample predictions using expanding windows.

---

### **7.1 The Procedure**

For each forecast origin $t = T_0, T_0+1, \ldots, T-1$:

1. Fit the model on data $\{X_1, \ldots, X_t\}$
2. Forecast $\hat{x}_{t+1|t}$
3. Observe the true $X_{t+1}$
4. Compute error: $e_{t+1} = X_{t+1} - \hat{x}_{t+1|t}$

---

### **7.2 Baselines Are Not Optional**

Every forecast evaluation must include baselines:

- **Naive:** $\hat{x}_{t+1|t} = X_t$ (random walk forecast)
- **Seasonal naive:** $\hat{x}_{t+1|t} = X_{t+1-s}$ (same month last year)

If your model cannot beat these, it provides no value.

---

### **7.3 Metrics**

$$
\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} e_i^2}, \qquad \text{MAE} = \frac{1}{N}\sum_{i=1}^{N} |e_i|
$$

> **Exam-safe statement:** "Rolling-origin backtesting is mandatory for honest forecast evaluation.
> Naive and seasonal naive baselines are minimum standards, not optional comparisons."


In [ ]:
# ============================================================
# PART 7: Rolling-Origin Backtest (Naive + Seasonal Naive + AR(1))
# ============================================================

np.random.seed(123)

# --- Simulate monthly PM2.5 (self-contained) ---
n = 240
period = 12
t_idx = np.arange(n)
idx = pd.date_range("2005-01-01", periods=n, freq="MS")

baseline = np.log(25)
trend_log = -0.0015 * t_idx
season_log = 0.25 * np.cos(2*np.pi*t_idx/period) - 0.15 * np.sin(2*np.pi*t_idx/period)
spike = np.zeros(n)
spike_months = np.random.choice(np.arange(12, n-12), size=10, replace=False)
spike[spike_months] = np.random.uniform(0.4, 0.9, size=len(spike_months))
eps = np.random.normal(0, 0.22, size=n)

log_pm25 = baseline + trend_log + season_log + spike + eps
y = pd.Series(log_pm25, index=idx, name="log_PM25")

# --- Rolling-origin backtest ---
def rolling_backtest(y, initial_train=120, season=12):
    y = y.dropna()
    preds = []

    # Initial AR(1) fit
    train = y.iloc[:initial_train]
    ar_res = ARIMA(train, order=(1, 0, 0)).fit()

    for t_end in range(initial_train, len(y) - 1):
        y_true = y.iloc[t_end + 1]

        # Naive
        fc_naive = y.iloc[t_end]

        # Seasonal naive
        if (t_end + 1 - season) >= 0:
            fc_snaive = y.iloc[t_end + 1 - season]
        else:
            fc_snaive = np.nan

        # AR(1) forecast (append without refit for speed)
        try:
            fc_obj = ar_res.get_forecast(steps=1)
            fc_ar1 = float(fc_obj.predicted_mean.iloc[0])
            ar_res = ar_res.append([y.iloc[t_end + 1]], refit=False)
        except Exception:
            fc_ar1 = np.nan

        preds.append({
            "time": y.index[t_end + 1],
            "actual": y_true,
            "naive": fc_naive,
            "seasonal_naive": fc_snaive,
            "ar1": fc_ar1,
        })

    return pd.DataFrame(preds).set_index("time")

bt = rolling_backtest(y, initial_train=120, season=12)
bt_clean = bt.dropna()

# --- Metrics ---
def rmse(a, f): return np.sqrt(np.mean((a - f)**2))
def mae(a, f): return np.mean(np.abs(a - f))

metrics = pd.DataFrame({
    "Model": ["Naive", "Seasonal Naive", "AR(1)"],
    "RMSE": [rmse(bt_clean["actual"], bt_clean["naive"]),
             rmse(bt_clean["actual"], bt_clean["seasonal_naive"]),
             rmse(bt_clean["actual"], bt_clean["ar1"])],
    "MAE":  [mae(bt_clean["actual"], bt_clean["naive"]),
             mae(bt_clean["actual"], bt_clean["seasonal_naive"]),
             mae(bt_clean["actual"], bt_clean["ar1"])],
})

print("=== Rolling-Origin Backtest (1-step) on log(PM2.5) ===")
print(f"N forecasts: {len(bt_clean)}")
print(metrics.to_string(index=False))

# --- Plot last 36 months ---
tail = bt_clean.tail(36)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Forecasts comparison
axes[0].plot(tail.index, tail["actual"], linewidth=2, color="#2c3e50", label="Actual")
axes[0].plot(tail.index, tail["naive"], linewidth=1.2, linestyle="--", label="Naive")
axes[0].plot(tail.index, tail["seasonal_naive"], linewidth=1.2, linestyle="--", label="Seasonal Naive")
axes[0].plot(tail.index, tail["ar1"], linewidth=1.2, linestyle="--", label="AR(1)")
axes[0].set_title("Rolling-Origin 1-Step Forecasts (last 36 months)", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Cumulative squared errors
for model, color, label in [("naive", "#e74c3c", "Naive"),
                              ("seasonal_naive", "#3498db", "Seasonal Naive"),
                              ("ar1", "#27ae60", "AR(1)")]:
    cse = np.cumsum((bt_clean["actual"] - bt_clean[model])**2)
    axes[1].plot(bt_clean.index, cse, linewidth=1.5, color=color, label=label)

axes[1].set_title("Cumulative Squared Error (lower is better)", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Seasonal naive typically wins for strongly seasonal data")
print("  - AR(1) without seasonal component cannot capture period-12 patterns")
print("  - This motivates SARIMA forecasting (Week 2's SARIMA models)")


## **PART 8 — Synthesis (Exam-Ready)**

> **Core points you must be able to state without hesitation:**

---

### **Forecasting Rules**

- $h$-step forecasting is **not** "repeat 1-step" — it has its own error variance structure
- For AR(1): $\hat{x}_{t+h|t} = \phi^h X_t$, point forecast decays to unconditional mean
- Forecast uncertainty grows with horizon via accumulated shocks ($\psi$-weights)
- Uncertainty saturates at $\gamma(0)$ (unconditional variance) for stationary processes

### **Prediction Intervals**

- PIs assume correct structure, stable variance, Gaussian shocks, and no structural breaks
- **Misspecification destroys interval validity** even when point forecasts look reasonable
- Passing a whiteness test does **not** guarantee correct coverage
- Heavy tails cause systematic **undercoverage**

### **$\psi$-weights**

- ARMA prediction intervals are justified through $\psi$-weights from the Wold representation
- For AR(1): $\psi_j = \phi^j$
- For ARMA(1,1): $\psi_0 = 1$, $\psi_j = \phi^{j-1}(\phi + \theta)$ for $j \geq 1$
- The MA component amplifies initial uncertainty

### **Evaluation**

- Rolling-origin backtesting is **mandatory** for honest forecast evaluation
- Naive and seasonal naive baselines are **minimum standards**, not optional
- A model that cannot beat naive provides no value
- Forecast evaluation must respect time order (no lookahead)


## **PART 9 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** What two objects define a forecast? Why is the forecast error variance as important as the point forecast?

**Q2.** Derive the $h$-step forecast error variance for AR(1). Why does it saturate at $\gamma(0)$?

**Q3.** What are $\psi$-weights? How do they connect the Wold decomposition to forecast uncertainty?

**Q4.** Give the $\psi$-weights for ARMA(1,1). How does $\psi_1 = \phi + \theta$ differ from the AR(1) case?

**Q5.** A colleague constructs prediction intervals from a model whose residuals pass the Ljung–Box test. Are the PIs necessarily valid? Explain.

**Q6.** You fit a Gaussian ARIMA model to financial returns data with heavy tails. What happens to your prediction interval coverage? Why?

**Q7.** Explain the difference between a naive forecast and a seasonal naive forecast. When would you expect seasonal naive to dominate?

**Q8.** Why must backtesting respect time order? What goes wrong with cross-validation in time series?

---

### **Computation Questions**

**Q9.** For AR(1) with $\phi = 0.8$, $\sigma^2 = 1$:
- Compute the 1-step, 5-step, and 20-step forecast error variance
- Compute the width of a 95% PI at each horizon
- At what horizon does $\mathrm{Var}(e_{t+h})$ reach 95% of $\gamma(0)$?

**Q10.** For ARMA(1,1) with $\phi = 0.5$, $\theta = 0.3$, $\sigma^2 = 1$:
- Compute $\psi_0, \psi_1, \psi_2, \psi_3$
- Compute $\mathrm{Var}(e_{t+3})$

**Q11.** Given the following backtest results:

| Model | RMSE | MAE |
|---|---|---|
| Naive | 0.35 | 0.28 |
| Seasonal Naive | 0.22 | 0.17 |
| ARIMA(1,1,1) | 0.24 | 0.19 |

Is the ARIMA model useful? Justify your answer.

---

### **Practical Challenge**

**Q12.** Modify the fan chart code (Part 4) to compare fan charts for $\phi = 0.3$ vs $\phi = 0.95$. Which provides informative forecasts at longer horizons? Why?

**Q13.** Add a SARIMA(1,1,1)(0,1,1)$_{12}$ forecaster to the rolling-origin backtest in Part 7. Does it beat seasonal naive?
